# 01 — Persiapan Data

Notebook ini menyiapkan data yang dipakai semua eksperimen. Tidak ada model yang
dilatih di sini.

Isi:
1. memuat data mentah dan memeriksa tipe/kelengkapan,
2. menguji parser angka untuk string seperti `68.0 m2`, `Rp 1`, dan `405.000.000`,
3. membersihkan baris mustahil, menormalkan nama wilayah, dan membuat target regresi/klasifikasi,
4. menyimpan split latih/uji dan lima lipatan CV yang akan dipakai ulang,
5. mencetak sidik jari data (jumlah baris dan SHA-256 baris latih/uji).

Keluaran ditulis ke `docs/perbandingan_output/`. Di Google Colab, modul dan data
diunduh otomatis dari GitHub dan keluaran berada di `/content/perbandingan_output/`.

In [1]:
import importlib.util
import os
import subprocess
import sys
import urllib.request
from pathlib import Path

import pandas as pd
from IPython.display import display

if importlib.util.find_spec("xgboost") is None:
    print("Memasang xgboost (diperlukan notebook model)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])

cwd = Path.cwd()
module_path = cwd / "perbandingan_common.py"
if not module_path.exists():
    docs_module = cwd / "docs" / "perbandingan_common.py"
    if docs_module.exists():
        sys.path.insert(0, str(docs_module.parent))
    else:
        url = "https://raw.githubusercontent.com/RobbyCatur/rumahpoint/main/docs/perbandingan_common.py"
        print("Mengunduh modul bersama dari GitHub ...")
        urllib.request.urlretrieve(url, module_path)
        sys.path.insert(0, str(cwd))
else:
    sys.path.insert(0, str(cwd))

import perbandingan_common as c

print("modul:", c.__file__)
print("repo terdeteksi:", c.REPO_ROOT)
print("direktori keluaran:", c.OUTPUT_DIR)
display(pd.DataFrame([c.describe_environment()]).T.rename(columns={0: "nilai"}))

modul: /opt/rumahpoint/docs/perbandingan_common.py
repo terdeteksi: /opt/rumahpoint
direktori keluaran: /opt/rumahpoint/docs/perbandingan_output


,nilai
python,3.12.3
platform,Linux-7.0.14-28-pve-x86_64-with-glibc2.39
sklearn,1.9.0
pandas,3.0.5
numpy,2.5.2
joblib,1.6.0
xgboost,3.4.1


## 1. Data mentah

Sumber tunggal adalah `dataset/properti_multi_kota.csv` (union enam sumber scraping).
Kolom yang dibaca: `sumber`, `lokasi`, lima fitur properti, dan `harga`.

In [2]:
raw = c.load_raw()
print(f"bentuk data mentah: {raw.shape[0]:,} baris x {raw.shape[1]} kolom")
display(raw.head(5))
ringkas = pd.DataFrame({
    "kolom": raw.columns,
    "tipe": raw.dtypes.astype(str).values,
    "kosong": raw.isna().sum().values,
})
display(ringkas)
display(raw["sumber"].value_counts().rename_axis("sumber").to_frame("baris").head(12))

bentuk data mentah: 64,650 baris x 8 kolom


,sumber,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan,harga
0,uas-properti/Jabodetabek,Tangerang,30.0,NaN,3.0,3.0,33.0,2.037543e+08
1,uas-properti/Jabodetabek,Bekasi,122.0,NaN,5.0,4.0,19.0,1.058032e+09
2,uas-properti/Jabodetabek,Bekasi,142.8,NaN,1.0,1.0,9.0,1.335793e+09
3,uas-properti/Jabodetabek,Bekasi,152.6,NaN,4.0,3.0,6.0,1.613771e+09
4,uas-properti/Jabodetabek,Depok,61.2,NaN,1.0,1.0,9.0,5.221562e+08


,kolom,tipe,kosong
0,sumber,str,0
1,lokasi,str,0
2,luas_m2,float64,0
3,luas_bangunan,float64,1604
4,jumlah_kamar,float64,396
5,jumlah_kamar_mandi,float64,408
6,usia_bangunan,float64,59861
7,harga,float64,0


,baris
sumber,
FLUS/komersil,39423
rumah123/Surabaya,2000
uas-properti/Jabodetabek,1530
99.co/Bekasi,400
99.co/Bogor,400
99.co/Tangerang,400
99.co/Medan,386
99.co/Malang,364
99.co/Bandar Lampung,338


## 2. Uji parser angka

Data scraping mencampur angka dan satuan. Parser yang sama dipakai untuk semua
kolom numerik. Kasus uji mencakup angka desimal, pemisah ribuan Indonesia,
notasi ilmiah, satuan `m2`, dan harga dalam juta/miliar. Jika satu kasus gagal,
notebook berhenti supaya kesalahan tidak terbawa ke perbandingan.

In [3]:
uji_parser = c.parse_number_selftest()
display(uji_parser)
assert uji_parser["lulus"].all(), "Parser angka gagal pada salah satu kasus uji"
print("Semua kasus uji parser lulus.")

,input,harapan,hasil,lulus
0,'68.0 m2',6.800000e+01,6.800000e+01,True
1,'Rp 1',1.000000e+00,1.000000e+00,True
2,'405.000.000',4.050000e+08,4.050000e+08,True
3,'1.200',1.200000e+03,1.200000e+03,True
4,"'1.234,56'",1.234560e+03,1.234560e+03,True
5,"'1,234.56'",1.234560e+03,1.234560e+03,True
6,'Rp 542 Juta',5.420000e+08,5.420000e+08,True
7,"'1,2 M'",1.200000e+09,1.200000e+09,True
8,'1.1111111111111112e+18',1.111111e+18,1.111111e+18,True
9,'',NaN,NaN,True


Semua kasus uji parser lulus.


## 3. Pembersihan data

Aturan dipilih untuk membuang nilai yang jelas bukan properti nyata, bukan untuk
menghapus harga ekstrem yang masih mungkin. Urutan saringan:

| Saringan | Alasan |
| --- | --- |
| lokasi kosong | tidak bisa dipetakan ke wilayah |
| harga/luas tidak bisa diparse atau tidak positif | bukan angka yang bisa dilatih |
| harga di bawah Rp 1 juta | menangkap placeholder seperti `Rp 1` |
| harga di atas Rp 1 kuadriliun | placeholder digit berulang |
| luas tanah di bawah 10 m² | batas yang sama dengan `ml/train.py` |
| luas tanah di atas 1.000.000 m² | daftar tanah, bukan rumah |
| harga per m² di atas Rp 100 juta | batas yang sama dengan `ml/train.py` |

Nilai fitur yang mustahil (luas bangunan di luar 1..100.000 m², kamar di luar
0..50) diubah menjadi kosong, lalu diimputasi median **di dalam pipeline** pada
setiap lipatan. Baris duplikat dihitung dan dilaporkan, tetapi tidak dibuang:
dua rumah bisa memiliki kombinasi fitur dan harga yang sama.

Kategori `Mahal/Murah` memakai median harga per kota pada seluruh data bersih,
mengikuti `database/seed.py`. Label ini tidak dipakai sebagai fitur.

In [4]:
df, rep = c.get_dataset()
funnel = pd.DataFrame([
    ("lokasi kosong", rep["drop_lokasi_kosong"]),
    ("harga tidak valid", rep["drop_harga_tidak_valid"]),
    ("luas tanah tidak valid", rep["drop_luas_tidak_valid"]),
    ("harga di bawah Rp 1 juta", rep["drop_harga_placeholder"]),
    ("harga di atas Rp 1 kuadriliun", rep["drop_harga_terlalu_besar"]),
    ("luas tanah di bawah 10 m2", rep["drop_luas_tanah_kecil"]),
    ("luas tanah di atas 1.000.000 m2", rep["drop_luas_tanah_besar"]),
    ("harga per m2 di atas Rp 100 juta", rep["drop_harga_per_m2"]),
    ("lokasi kanonik kosong", rep["drop_lokasi_kanonik_kosong"]),
], columns=["saringan", "baris_dibuang"])
display(funnel)
print(
    f"mentah {rep['raw_rows']:,} -> bersih {rep['clean_rows']:,} "
    f"({rep['clean_rows'] / rep['raw_rows'] * 100:.2f}%); total dibuang {rep['drop_total']:,}"
)
ringkas = pd.DataFrame([
    ("baris identik (tidak dibuang)", rep["baris_identik"]),
    ("kota dengan satu baris", rep["kota_dengan_satu_baris"]),
    ("median luas bangunan", rep["median_luas_bangunan"]),
    ("median usia bangunan (tahun)", rep["median_usia_bangunan_norm"]),
], columns=["ringkasan", "nilai"])
display(ringkas)
display(df[["sumber", "lokasi_mentah", "lokasi", "luas_m2", "luas_bangunan",
            "jumlah_kamar", "jumlah_kamar_mandi", "usia_bangunan_norm", "harga",
            "kategori_harga"]].head(5))

,saringan,baris_dibuang
0,lokasi kosong,0
1,harga tidak valid,0
2,luas tanah tidak valid,0
3,harga di bawah Rp 1 juta,245
4,harga di atas Rp 1 kuadriliun,14
5,luas tanah di bawah 10 m2,39
6,luas tanah di atas 1.000.000 m2,0
7,harga per m2 di atas Rp 100 juta,124
8,lokasi kanonik kosong,0


mentah 64,650 -> bersih 64,228 (99.35%); total dibuang 422


,ringkasan,nilai
0,baris identik (tidak dibuang),7144.0
1,kota dengan satu baris,47.0
2,median luas bangunan,64.0
3,median usia bangunan (tahun),9.0


,sumber,lokasi_mentah,lokasi,luas_m2,luas_bangunan,jumlah_kamar,jumlah_kamar_mandi,usia_bangunan_norm,harga,kategori_harga
0,uas-properti/Jabodetabek,Tangerang,Tangerang,30.0,NaN,3.0,3.0,33.0,2.037543e+08,Murah
1,uas-properti/Jabodetabek,Bekasi,Bekasi,122.0,NaN,5.0,4.0,19.0,1.058032e+09,Mahal
2,uas-properti/Jabodetabek,Bekasi,Bekasi,142.8,NaN,1.0,1.0,9.0,1.335793e+09,Mahal
3,uas-properti/Jabodetabek,Bekasi,Bekasi,152.6,NaN,4.0,3.0,6.0,1.613771e+09,Mahal
4,uas-properti/Jabodetabek,Depok,Depok,61.2,NaN,1.0,1.0,9.0,5.221562e+08,Murah


## 4. Normalisasi nama wilayah

`mobile/assets/data/wilayah.json` berisi 34 provinsi dan daftar kabupaten/kota
yang bisa dipilih pengguna. Aturan pemetaan:

1. nama yang sudah persis ada di aset dipertahankan;
2. `Kota X` dicoba sebagai `X` karena aset tidak memakai awalan `Kota `;
3. nama tanpa awalan dicoba sebagai `Kabupaten X`;
4. nama yang tetap tidak ditemukan dibiarkan apa adanya dan dilaporkan.

Nama mentah tetap disimpan di `lokasi_mentah` agar asal-usulnya bisa diperiksa.
Nama yang tidak ada di aset tidak bisa dipilih lewat aplikasi; barisnya tetap
dihitung di data latih sebagai kategori tersendiri.

In [5]:
print(f"lokasi mentah unik   : {rep['lokasi_mentah_unik']:,}")
print(f"lokasi kanonik unik  : {rep['lokasi_kanonik_unik']:,}")
print(f"baris yang dipetakan : {rep['lokasi_baris_dipetakan']:,}")
display(pd.DataFrame(rep["contoh_pemetaan"]))
tidak_ada = pd.DataFrame({"lokasi": rep["lokasi_tidak_ada_di_aset"]})
print(f"nama yang tidak ada di aset: {len(tidak_ada):,}")
display(tidak_ada)

lokasi mentah unik   : 593
lokasi kanonik unik  : 445
baris yang dipetakan : 7,787


,lokasi_mentah,lokasi,baris
0,Sidoarjo,Kabupaten Sidoarjo,320
1,Klaten,Kabupaten Klaten,297
2,Tabanan,Kabupaten Tabanan,297
3,Badung,Kabupaten Badung,293
4,Sukoharjo,Kabupaten Sukoharjo,222
5,Bandung Barat,Kabupaten Bandung Barat,212
6,Boyolali,Kabupaten Boyolali,203
7,Gresik,Kabupaten Gresik,196
8,Cianjur,Kabupaten Cianjur,194
9,Sumedang,Kabupaten Sumedang,181


nama yang tidak ada di aset: 57


,lokasi
0,Aceh
1,Bali
2,Banjarbaru
3,Banten
4,Banyuasin
5,Bau Bau
6,Daerah Istimewa Yogyakarta
7,Dki Jakarta
8,Dumai
9,Gunung Kidul


## 5. Distribusi target

Regresi memakai `log10(harga)`. Klasifikasi memakai label biner
`Mahal` bila harga suatu baris ≥ median kota kanoniknya. Distribusi kelas
mendekati seimbang karena ambangnya median per kota, bukan median nasional.

Ambang median per kota dihitung dari seluruh data bersih, termasuk baris uji,
mengikuti `database/seed.py`; label klasifikasi tidak sepenuhnya out-of-sample.

In [6]:
display(df["kategori_harga"].value_counts().rename_axis("kategori").to_frame("baris"))
per_kota = df.groupby("lokasi").size()
display(per_kota.describe().to_frame("baris_per_kota").T)
display(df["sumber"].value_counts().head(10).rename_axis("sumber").to_frame("baris"))

,baris
kategori,
Mahal,32690
Murah,31538


,count,mean,std,min,25%,50%,75%,max
baris_per_kota,445.0,144.332584,344.170457,1.0,4.0,22.0,112.0,2310.0


,baris
sumber,
FLUS/komersil,39120
rumah123/Surabaya,1995
uas-properti/Jabodetabek,1528
99.co/Bekasi,400
99.co/Bogor,400
99.co/Tangerang,399
99.co/Medan,374
99.co/Malang,364
99.co/Bandar Lampung,329


## 6. Kelengkapan usia bangunan

Usia bangunan hanya diuji sebagai bukti tambahan di notebook regresi, bukan
menjadi fitur utama. Alasannya: sebagian besar baris tidak punya usia, sumber
terbesar tidak menyediakannya, dan data scraping lama berasal dari snapshot
yang berbeda. Normalisasi memakai tahun acuan tetap 2026: nilai 1900..2026
diubah menjadi umur, nilai 0..126 dianggap sudah umur, dan nilai lain menjadi
kosong untuk diimputasi median bila fitur itu diuji.

In [7]:
usia_sumber = pd.DataFrame(rep["usia_per_sumber"])
display(usia_sumber.head(12))
print(f"jumlah kelompok sumber: {len(usia_sumber):,}")
tanpa = usia_sumber[usia_sumber["usia_ada"] == 0]
print(f"kelompok tanpa satu pun usia: {len(tanpa):,}; baris: {tanpa['baris'].sum():,}")
print(f"usia mentah kosong: {df['usia_bangunan'].isna().sum():,} "
      f"({df['usia_bangunan'].isna().mean() * 100:.2f}%)")
print(f"usia ternormalisasi kosong: {df['usia_bangunan_norm'].isna().sum():,} "
      f"({df['usia_bangunan_norm'].isna().mean() * 100:.2f}%)")
print(f"tahun diubah menjadi umur: {rep['usia_tahun_ke_umur']:,}; "
      f"nilai mustahil -> kosong: {rep['usia_nilai_mustahil_nan']:,}")

,sumber,baris,usia_ada,usia_persen
0,FLUS/komersil,39120,0,0.00
1,rumah123/Surabaya,1995,931,46.67
2,uas-properti/Jabodetabek,1528,1490,97.51
3,99.co/Bogor,400,64,16.00
4,99.co/Bekasi,400,61,15.25
5,99.co/Tangerang,399,68,17.04
6,99.co/Medan,374,43,11.50
7,99.co/Malang,364,96,26.37
8,99.co/Bandar Lampung,329,21,6.38
9,99.co/Cirebon,312,48,15.38


jumlah kelompok sumber: 511
kelompok tanpa satu pun usia: 362; baris: 46,609
usia mentah kosong: 59,443 (92.55%)
usia ternormalisasi kosong: 59,489 (92.62%)
tahun diubah menjadi umur: 3,070; nilai mustahil -> kosong: 46


## 7. Split dan lipatan CV

Satu split hold-out 80/20 dengan `random_state=42` dipakai untuk regresi dan
klasifikasi. Lima lipatan `KFold(n_splits=5, shuffle=True, random_state=42)`
dibentuk hanya dari baris latih. Praproses selalu di-fit di dalam pipeline pada
data latih lipatan, sehingga tidak ada kebocoran dari data uji.

Split dan lipatan disimpan sebagai indeks posisi. Notebook lain memuat file yang
sama dan memeriksa SHA-256 baris, bukan membentuk split baru.

In [8]:
split, train_idx, test_idx = c.ensure_split(df)
folds = c.ensure_folds(len(train_idx))
fp = c.ensure_fingerprint(df, rep, split, folds)
c.print_fingerprint(fp)
lipatan = pd.DataFrame([
    {
        "fold": f["fold"],
        "n_train": f["n_train"],
        "n_test": f["n_test"],
        "hash_indeks_uji": f["test_index_sha256"],
    }
    for f in folds
])
display(lipatan)
print(f"baris latih: {len(train_idx):,}; baris uji: {len(test_idx):,}")

=== SIDIK JARI DATA ===
berkas dataset        : /opt/rumahpoint/dataset/properti_multi_kota.csv
baris mentah          : 64,650
baris bersih          : 64,228
baris latih / uji     : 51,382 / 12,846
hash baris latih      : cfe3defcae087f44fa6a7189a319ef258f4138318a7532b315bf80a85dc9990e
hash baris uji        : 718f7f914fe83904e4899da30d2fc6b7a06e249431befe26985f93e43efe7aef
hash file dataset     : 34375406a6d72b21bad55e5784caddc931a85bb20c999414d3fdd6b2f5774d18
lipatan 0 (indeks uji): f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5d6798761cf3ae938d4
lipatan 1 (indeks uji): c177e4c57ee746c9d021815cecfbc4e8128469d08d1617f086033940cee7efe6
lipatan 2 (indeks uji): 7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21beddf20741479178c025e
lipatan 3 (indeks uji): 9956fdc032bc252aeb7d0ecac5531626bac860dee609478a49110408ee8b19cd
lipatan 4 (indeks uji): a0b04506e601150298462828e0306e977fd83ad22063aa0ebd10d9663b7c51dc


,fold,n_train,n_test,hash_indeks_uji
0,0,41105,10277,f298c1b2ee29f320d674c00a96af6e48ce0dff211e66a5...
1,1,41105,10277,c177e4c57ee746c9d021815cecfbc4e8128469d08d1617...
2,2,41106,10276,7c6dfcd5824b61717f80eeddde5371bb4c93adb3a21bed...
3,3,41106,10276,9956fdc032bc252aeb7d0ecac5531626bac860dee60947...
4,4,41106,10276,a0b04506e601150298462828e0306e977fd83ad22063aa...


baris latih: 51,382; baris uji: 12,846


## 8. Berkas yang disimpan

Tiga berkas ini menjadi kontrak antar-notebook. Jangan menghapusnya selama
eksperimen; jika data berubah, hapus ketiganya dan jalankan ulang notebook ini.

In [9]:
for nama in ["split.json", "folds.json", "fingerprint.json"]:
    path = c.OUTPUT_DIR / nama
    print(f"{path} -> {path.exists()}")
print("dataset:", c.CSV_PATH)
print("aset wilayah:", c.ASSET_PATH)

/opt/rumahpoint/docs/perbandingan_output/split.json -> True
/opt/rumahpoint/docs/perbandingan_output/folds.json -> True
/opt/rumahpoint/docs/perbandingan_output/fingerprint.json -> True
dataset: /opt/rumahpoint/dataset/properti_multi_kota.csv
aset wilayah: /opt/rumahpoint/mobile/assets/data/wilayah.json


## Catatan Colab

Jika `repo terdeteksi` bernilai `None`, notebook berjalan di Colab. Modul dan
data diunduh dari `raw.githubusercontent.com/RobbyCatur/rumahpoint/main`, dan
keluaran disimpan di `/content/perbandingan_output`. Setelah semua notebook
selesai, unduh folder tersebut agar hasil bisa dilampirkan.